In [9]:
import torch 
import torch.nn as nn

In [10]:
GPT_CONFIG = {
    "vocab_size" : 50257,
    "context_length" : 128, 
    "emb_dim" : 768,
    "n_heads" : 12,
    "n_layers" : 12,
    "drop_rate" : 0.1,
    "qkv_bias" : False
}

In [11]:
from transformer import TransformerBlock 

x = torch.rand(2,4,768)
block = TransformerBlock(GPT_CONFIG)
output = block(x)

print(output)

tensor([[[ 0.6634, -0.2382,  1.6527,  ...,  0.5993,  0.9225,  0.6229],
         [ 0.6598,  1.4885, -0.1499,  ...,  0.2475, -0.1982,  1.2933],
         [-0.2272,  1.6436,  1.1978,  ...,  1.4538,  0.0816,  0.6833],
         [ 0.2439,  0.3920,  0.2285,  ..., -0.0127,  0.5418,  0.7124]],

        [[ 0.5753,  0.0814,  0.1852,  ...,  0.4911,  1.0234,  0.4973],
         [ 1.0569,  0.8737, -0.0492,  ..., -0.6923, -0.5545,  0.6613],
         [ 0.5969,  1.4523,  1.4645,  ...,  0.2814,  0.2855,  0.1815],
         [ 0.5463,  1.3250,  0.9420,  ...,  0.0018,  0.2722,  0.8356]]],
       grad_fn=<AddBackward0>)


In [12]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [13]:
from mini_GPT import GPTModel

tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

GPT2 = GPTModel(GPT_CONFIG)
output = GPT2(batch)

print(output.shape)


total_parameters = sum(p.numel() for p in GPT2.parameters())

print(f"Total Parameters : {total_parameters}")


print(f"Embedding layer Shape : {GPT2.token_emb.weight.shape}")
print(f"Output layer Shape : {GPT2.out_head.weight.shape}")

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])
torch.Size([2, 4, 50257])
Total Parameters : 162321408
Embedding layer Shape : torch.Size([50257, 768])
Output layer Shape : torch.Size([50257, 768])


In [ ]:
## Model without training 

from mini_GPT import GenerateText

device = "cuda" if torch.cuda.is_available() else "cpu"

start_context = "The artist looked"
encoded = tokenizer.encode(start_context)

print("----------------------------------------------")
print(f"Text: {start_context}")
print(f"Encoded tokens: {encoded}")

encoder_tensor = torch.tensor(encoded, dtype=torch.long).unsqueeze(0).to(device)

GPT2 = GPT2.to(device)
GPT2.eval()

out = GenerateText(
    model=GPT2,
    idx=encoder_tensor,
    max_new_tokens=50,
    context_size=GPT_CONFIG["context_length"]
)

decoded_text = tokenizer.decode(out.squeeze(0).cpu().tolist())

print(decoded_text)

----------------------------------------------
Text: The artist looked
Encoded tokens: [464, 6802, 3114]
The artist lookedlore Iran Indy Staff blush Staff blush referencing geopolitical withdrew service uncommon blushlucent reverted Sydney printf pets Eye Eye�� matchup tolerant Ivanka blush Buch� emphasizes emphasizes emphasizes emphasizes blush Buch��� uncommon blush Staff� MIDI emphasizes emphasizes emphasizes reverted matchup cognitive blush Ivanka


In [ ]:
### Model Trained on the the-verdict.txt data

model = GPTModel(GPT_CONFIG)
model.load_state_dict(torch.load("checkpoints/best_model.pt"))
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 162,321,408


In [21]:
### Model Trained on the the-verdict.txt data
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)
model.eval()

prompt = "The artist looked"

input_ids = tokenizer.encode(prompt)
input_ids = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)

output_ids = generate_text(
    model=model,
    input_ids=input_ids,
    max_new_tokens=100,
    context_length=GPT_CONFIG["context_length"],
    temperature=0.8,
    top_k=50
)

output_text = tokenizer.decode(output_ids[0].cpu().tolist())

print(output_text)

The artist looked about Victor Grindle was high above the easel placed so--that was having on Mrs. Gisburn had begun to have mentioned that there had begun to have mentioned that point the hot-house of their savour? No--one of mediocrity" (I have mentioned that Mrs. It was glad to was having on him up; and it was immediately perceptible that Mrs. It was rich; and it was having on him insignificant and was immediately perceptible that Mrs. It was
